# Where does one output come from?

Start with a small matrix product. Predict one answer, highlight its source cells,
and then compare the same calculation with a sum, a mean, and einsum.
Each figure answers one question.

In [ ]:
import numpy as np

import rainbow_tensor as rt

a = np.arange(6).reshape(2, 3)
b = np.arange(12).reshape(3, 4)
rt.shape(a)

## Predict the entry at row 1, column 2

Its source values are row `[3, 4, 5]` and column `[2, 6, 10]`.
The answer is `3 * 2 + 4 * 6 + 5 * 10 = 80`.
Change `focus` below to explore another output.

In [ ]:
visual = rt.matmul(a, b, focus=(1, 2))
assert (a @ b)[1, 2] == 80
visual

## Inspect the explanation in Python

A trace term contains source references multiplied together. Terms are added.
The trace reports how many terms exist and whether its stored sample is complete.

In [ ]:
print(visual.text)
print(visual.trace)

## Sum and mean use the same source group

Focus on the last row with the negative coordinate `(-1,)`.
A sum adds `3 + 4 + 5`. A mean divides that sum by `3`.

In [ ]:
rt.sum(a, axis=1, focus=(-1,))

In [ ]:
rt.mean(a, axis=1, focus=(-1,))

## Express the matrix product with labels

In `ij,jk->ik`, `i` is the output row and `k` is the output column.
The shared label `j` is summed out.

In [ ]:
rt.einsum("ij,jk->ik", a, b, focus=(1, 2))

## Shape and storage answer different questions

Transposition changes how indices reach the same stored values.
Compare byte strides and ownership before and after making a copy.

In [ ]:
rt.memory(a.T)

In [ ]:
rt.memory(a.T.copy())

## Preview a large selection without allocating the tensor

A shape tuple supplies placeholder values. The selection stores ranges and
can count a trillion coordinates without expanding them into a list.

In [ ]:
large = rt.index((1_000_000, 1_000_000), (Ellipsis,))
assert large.selected.count == 1_000_000_000_000
print(large.selected.count)
large

## Keep expensive arithmetic explicit

The display budget limits visible cells. A separate `max_terms` budget limits
arithmetic terms per output. A question mark means the value was not evaluated,
not that the result is zero or an approximate partial sum.
Use `max_terms=None` only when full evaluation is wanted.

In [ ]:
rt.sum((2, 100_000), axis=1, focus=(1,))

In [ ]:
rt.sum(a, axis=1, focus=(1,), max_terms=None)